# Kapitan 03: the Kubernetes generator

`kgenlib` turns `parameters.components.<name>` into Deployments, StatefulSets, Services,
ConfigMaps, Secrets, NetworkPolicies and more. The inventory is the only thing you write.


In [ ]:
cd /source/work/kapitan-reference
export HOME=/tmp
sed -n 1,60p inventory/classes/components/echo-server.yml


In [ ]:
cd /source/work/kapitan-reference
kapitan compile -t echo-server 2>&1 | tail -1 && ls compiled/echo-server/manifests && yq '.kind' compiled/echo-server/manifests/echo-server-bundle.yml


Now a component of our own: a new target file is enough. `type` defaults to deployment; a `service:` key makes the generator emit a Service for the ports that declare `service_port`; env, secrets, config maps, volumes and policies are all keys of the same map.


In [ ]:
cd /source/work/kapitan-reference
cat > inventory/targets/tutorials/hello.yml <<'YAML'
classes:
  - common

parameters:
  components:
    hello:
      image: traefik/whoami:v1.11.0
      replicas: 2
      service:
        type: ClusterIP
      ports:
        http:
          container_port: 80
          service_port: 80
      env:
        WHOAMI_NAME: hello from kapitan
        API_TOKEN:
          secretKeyRef:
            key: api-token
      secrets:
        secrets:
          data:
            api-token:
              value: ?{plain:targets/${target_name}/api-token||random:str:24|base64}
      resources:
        requests: { cpu: 20m, memory: 16Mi }
YAML
kapitan compile -t hello 2>&1 | tail -1 && ls compiled/hello/manifests


In [ ]:
cd /source/work/kapitan-reference
yq 'select(.kind == "Deployment") | .spec.replicas, .spec.template.spec.containers[0].env' compiled/hello/manifests/hello-bundle.yml; yq 'select(.kind == "Service") | .spec.ports' compiled/hello/manifests/hello-service.yml


Change the context, recompile, diff: this is the loop a platform team lives in. The generated README documents the component too.


In [ ]:
cd /source/work/kapitan-reference
cp -r compiled/hello /tmp/hello-before && sed -i 's/replicas: 2/replicas: 4/' inventory/targets/tutorials/hello.yml && kapitan compile -t hello 2>&1 | tail -1 && diff -r /tmp/hello-before compiled/hello || true


In [ ]:
cd /source/work/kapitan-reference
grep -A6 '## Components' compiled/hello/README.md | head -12
